In [11]:
# ============================================================
# Sampling Sufficiency (No Retraining) — Full, High-Resolution
# Implements:
#  1) Finer fraction grids + Monte-Carlo repeats
#  2) Stratified thinning by latitude bands, sectors, and season
#  3) Time-block cross-validation
#  4) Sector-wise trade-offs (global impact + sector-local)
# Truth: SOCAT+BGC-Argo MLP predictions; SOCAT/float only for masks
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyseaflux
from scipy.stats import linregress
import os

# --------------------------
# USER PATHS (edit as needed)
# --------------------------
PRED_SOCAT_AND_FLOATS = "/home/ldelaigue/Documents/Python/SOCAT-OSSE-SO/PREDICTION_MATRIX/matrix_predictions/pco2_prediction_matrix_SOCAT_AND_FLOATS_models.csv"
SOCAT_FILE            = "/home/ldelaigue/Documents/Python/SOCAT-OSSE-SO/MODEL_SOCAT_ONLY_OBS/processing_steps/SOCATv2025_SO_clean_SLA_RRS_CHL_WIND_PAR_CO2ATM_MLD_COORDS_PCO2.csv"
FLOATS_FILE           = "/home/ldelaigue/Documents/Python/SOCAT-OSSE-SO/MODEL_FLOATS/processing_steps/BGC-ARGO_mean_0-10m_surface_pco2_all_profiles_US_pco2_SLA_RRS_CHL_PAR_WIND_CO2ATM_MLD_COORDS.csv"
OUTDIR                = "/home/ldelaigue/Documents/Python/SOCAT-OSSE-SO/SAMPLING_SUFFICIENCY"

os.makedirs(OUTDIR, exist_ok=True)

# -----------------
# CONSTANTS & AREA
# -----------------
OCEAN_AREA_M2 = 7e13  # Southern Ocean south of 35°S (your paper)
R = 6371000.0
deg2rad = np.pi / 180.0
lat_centers = np.arange(-89.5, 90.5, 1.0)[:-1]
band_areas = (2*np.pi*R**2) * (np.sin((lat_centers+0.5)*deg2rad) - np.sin((lat_centers-0.5)*deg2rad))
cell_area_by_lat = pd.Series(band_areas/360.0, index=np.arange(-89, 90, 1))  # m^2 per 1° cell

# Sectors (lon in degrees, -180..180)
SECTORS = {
    'Atlantic': (-70,  20),
    'Indian':   ( 20, 150),
    'Pacific':  (150, -70)  # wraps the dateline: lon >=150 or lon <= -70
}

# --------------
# UTIL FUNCTIONS
# --------------
def add_area_weights_grid(df):
    return df.merge(cell_area_by_lat.rename('cell_area_m2'),
                    left_on='lat_bin', right_index=True, how='left')

def ensure_ensemble_col(df, ensemble_col='ensemble_pco2_pred'):
    """Create ensemble pCO2 column by averaging pco2_pred_model_* if not present."""
    if ensemble_col in df.columns:
        return df
    model_cols = [c for c in df.columns if c.startswith('pco2_pred_model_')]
    if not model_cols:
        raise ValueError("No per-model pCO2 columns (pco2_pred_model_*) and no ensemble column present.")
    out = df.copy()
    out[ensemble_col] = out[model_cols].mean(axis=1)
    return out

def compute_co2_flux_from_predictions(df):
    wind_speed = df["wind_speed"].values
    temp_C     = df["temperature"].values
    salinity   = df["salinity"].values
    pCO2_sw    = df["ensemble_pco2_pred"].values
    pCO2_air   = df["atm_co2"].values
    pres_hPa   = df["air_pressure_hpa"].values if "air_pressure_hpa" in df else 1013.25
    k_w_cmhr   = pyseaflux.gas_transfer_velocity.k_Ho06(wind_speed, temp_C)
    flux_mol_m2_day = pyseaflux.flux_calculations.flux_bulk(temp_C, salinity, pCO2_sw, pCO2_air, pres_hPa, k_w_cmhr)
    return flux_mol_m2_day * 12.01  # g C m^-2 day^-1

def basin_mean_annual_from_grid_month(df_grid):
    """Area-weighted annual mean from grid-month flux (g C m^-2 month)."""
    g = add_area_weights_grid(df_grid)
    yearly = (g.groupby(['year'])
                .apply(lambda d: (d['flux_gC_m2_month'] * d['cell_area_m2']).sum()
                                / d['cell_area_m2'].sum())
                .reset_index(name='aw_flux_gC_m2_month'))
    yearly['aw_flux_gC_m2_yr'] = yearly['aw_flux_gC_m2_month'] * 12.0
    return yearly[['year','aw_flux_gC_m2_yr']]

def line_trend(year, y):
    m = ~np.isnan(y)
    if m.sum() < 3:
        return np.nan, np.nan
    lr = linregress(year[m], y[m])
    return lr.slope, lr.pvalue  # slope in g C m^-2 yr^-2

def eval_scenario(truth_annual, scenario_annual, yr0=2003, yr1=2024):
    ta = truth_annual[(truth_annual['year']>=yr0)&(truth_annual['year']<=yr1)].copy()
    sa = scenario_annual[(scenario_annual['year']>=yr0)&(scenario_annual['year']<=yr1)].copy()
    if sa.empty:
        return np.nan, np.nan, ta.rename(columns={'Truth':'Scenario'})
    # standardize col name
    scen_cols = [c for c in sa.columns if c != 'year']
    scen_col = 'aw_flux_gC_m2_yr' if 'aw_flux_gC_m2_yr' in scen_cols else scen_cols[0]
    sa = sa.rename(columns={scen_col:'Scenario'})
    df = ta.merge(sa, on='year', how='left')
    mean_bias = (df['Scenario'] - df['Truth']).mean()  # g C m^-2 yr^-1
    t_truth, _ = line_trend(df['year'].values, df['Truth'].values)
    t_scen,  _ = line_trend(df['year'].values, df['Scenario'].values)
    trend_bias = t_scen - t_truth  # g C m^-2 yr^-2
    return mean_bias, trend_bias, df

def to_pg(x):   # g C m^-2 yr^-1 (or yr^-2) → Pg C yr^-1 (or yr^-2)
    return None if pd.isna(x) else x*OCEAN_AREA_M2/1e15

# -------------------------------
# 1) LOAD TRUTH (SOCAT+FLOATS MLP)
# -------------------------------
pred = pd.read_csv(PRED_SOCAT_AND_FLOATS)
pred = ensure_ensemble_col(pred)
pred['time'] = pd.to_datetime(pred['time'])
pred['year'] = pred['time'].dt.year
pred['month'] = pred['time'].dt.month
pred['lat_bin'] = pred['latitude'].astype(int)
pred['lon_bin'] = pred['longitude'].astype(int)

# Flux at grid-time points → grid-month
pred['flux_gC_m2_day']   = compute_co2_flux_from_predictions(pred)
pred['flux_gC_m2_month'] = pred['flux_gC_m2_day'] * pred['time'].dt.days_in_month
truth_grid = (pred.groupby(['lat_bin','lon_bin','year','month'])['flux_gC_m2_month']
                .mean().reset_index())

truth_annual = basin_mean_annual_from_grid_month(truth_grid).rename(columns={'aw_flux_gC_m2_yr':'Truth'})

# ---------------------------
# 2) BUILD REAL-WORLD MASKS
# ---------------------------
def prep_obs_footprint(df, time_col):
    tmp = df.copy()
    tmp['time'] = pd.to_datetime(tmp[time_col])
    tmp['year'] = tmp['time'].dt.year
    tmp['month'] = tmp['time'].dt.month
    tmp['lat_bin'] = tmp['latitude'].astype(int)
    tmp['lon_bin'] = tmp['longitude'].astype(int)
    fp = (tmp.groupby(['lat_bin','lon_bin','year','month'])
             .size().reset_index(name='n'))
    fp['mask'] = 1
    return fp[['lat_bin','lon_bin','year','month','mask']]

# Load SOCAT & floats ONLY for footprint masks
socat_obs  = pd.read_csv(SOCAT_FILE, low_memory=False)
floats_obs = pd.read_csv(FLOATS_FILE, low_memory=False)
socat_fp   = prep_obs_footprint(socat_obs,  time_col='date' if 'date' in socat_obs.columns else 'datetime')
floats_fp  = prep_obs_footprint(floats_obs, time_col='datetime')

def month_to_season(m):
    if m in [12,1,2]:  return 'summer'
    if m in [3,4,5]:   return 'autumn'
    if m in [6,7,8]:   return 'winter'
    return 'spring'

def thin_footprint(fp, frac=1.0, by_season=True, rng=42):
    """Randomly keep a fraction of footprint (grid-months). If by_season: thin within each season."""
    np.random.seed(rng)
    f = fp.copy()
    f['season'] = f['month'].apply(month_to_season)
    f['rand']   = np.random.rand(len(f))
    if by_season:
        keep_idx = []
        for s in ['summer','autumn','winter','spring']:
            sel = (f['season']==s)
            if sel.any():
                thr = np.quantile(f.loc[sel,'rand'], 1-frac)
                keep_idx.append(f.index[sel & (f['rand'] <= thr)])
        if keep_idx:
            keep_idx = np.concatenate(keep_idx)
            return f.loc[keep_idx, ['lat_bin','lon_bin','year','month','mask']]
        return f.iloc[0:0][['lat_bin','lon_bin','year','month','mask']]
    else:
        thr = np.quantile(f['rand'], 1-frac)
        return f.loc[f['rand'] <= thr, ['lat_bin','lon_bin','year','month','mask']]

def apply_spatiotemporal_filters(fp, drop_winter=False, keep_south_of=None, drop_sector=None):
    """drop_winter: True→remove JJA; keep_south_of: latitude; drop_sector: (lon_min, lon_max) (wrap ok)."""
    f = fp.copy()
    if drop_winter:
        f = f[~f['month'].isin([6,7,8])]
    if keep_south_of is not None:
        f = f[f['lat_bin'] <= keep_south_of]
    if drop_sector is not None:
        lon_min, lon_max = drop_sector
        if lon_min <= lon_max:
            f = f[~((f['lon_bin'] >= lon_min) & (f['lon_bin'] <= lon_max))]
        else:
            f = f[~((f['lon_bin'] >= lon_min) | (f['lon_bin'] <= lon_max))]
    return f

def sector_name_from_lon(lon):
    if (-70 <= lon <= 20):  return 'Atlantic'
    if (20 < lon <= 150):   return 'Indian'
    # Pacific wraps: lon >=150 or lon <= -70
    return 'Pacific'

def thin_stratified(fp, lat_bands, season_fracs, sector_fracs=None, rng=42):
    """
    Vectorized stratified thinning.
    lat_bands: list of (lat_min, lat_max, frac_lat) — applied by row membership.
    season_fracs: dict like {'summer':1.0,'autumn':0.6,'winter':1.0,'spring':0.5}
    sector_fracs: dict like {'Atlantic':0.6,'Indian':1.0,'Pacific':0.4} (optional)
    Keeps a row if rand <= final_fraction where final_fraction = min(frac_lat, frac_season, frac_sector).
    """
    np.random.seed(rng)
    f = fp.copy()
    f['season'] = f['month'].apply(month_to_season)
    f['sector'] = f['lon_bin'].apply(sector_name_from_lon)
    f['rand']   = np.random.rand(len(f))

    # start with 1.0 everywhere
    frac_lat = np.ones(len(f), dtype=float)
    for (la, lb, fl) in lat_bands:
        sel = (f['lat_bin'] >= la) & (f['lat_bin'] <= lb)
        frac_lat[sel] = np.minimum(frac_lat[sel], fl)

    frac_season = f['season'].map(season_fracs).fillna(1.0).values
    if sector_fracs is not None:
        frac_sector = f['sector'].map(sector_fracs).fillna(1.0).values
        final_frac = np.minimum(np.minimum(frac_lat, frac_season), frac_sector)
    else:
        final_frac = np.minimum(frac_lat, frac_season)

    keep = f['rand'].values <= final_frac
    return f.loc[keep, ['lat_bin','lon_bin','year','month','mask']]

# -------------------------------
# 3) MASK TRUTH & METRICS
# -------------------------------
def masked_basin_annual(truth_grid, masks):
    """Union masks → sample truth only where covered → annual basin mean."""
    if len(masks)==0:
        return pd.DataFrame(columns=['year','aw_flux_gC_m2_yr'])
    mask_union = pd.concat(masks, ignore_index=True)
    mask_union = (mask_union.groupby(['lat_bin','lon_bin','year','month'])
                            .size().reset_index()[['lat_bin','lon_bin','year','month']])
    obs_truth = truth_grid.merge(mask_union, on=['lat_bin','lon_bin','year','month'], how='inner')
    if obs_truth.empty:
        return pd.DataFrame(columns=['year','aw_flux_gC_m2_yr'])
    return basin_mean_annual_from_grid_month(obs_truth)

# ======================================================
# A) FINER FRACTIONS + MONTE-CARLO (global trade-offs)
# ======================================================
fractions = np.linspace(0.05, 1.00, 37)  # finer steps
n_reps = 20                              # Monte Carlo repeats

records = []
for f in fractions:
    for rep in range(n_reps):
        # SOCAT curve: thin SOCAT, floats full
        soc_thin = thin_footprint(socat_fp, frac=f, by_season=True, rng=1000 + rep)
        scen = masked_basin_annual(truth_grid, [soc_thin, floats_fp])
        mb, tb, _ = eval_scenario(truth_annual, scen)
        records.append({'who':'SOCAT','frac':f,'rep':rep,
                        'mb_pg':to_pg(mb),'tb_pg':to_pg(tb)})

        # Floats curve: thin floats, SOCAT full
        fl_thin = thin_footprint(floats_fp, frac=f, by_season=True, rng=2000 + rep)
        scen = masked_basin_annual(truth_grid, [socat_fp, fl_thin])
        mb, tb, _ = eval_scenario(truth_annual, scen)
        records.append({'who':'Floats','frac':f,'rep':rep,
                        'mb_pg':to_pg(mb),'tb_pg':to_pg(tb)})

tradeoff_mc = pd.DataFrame.from_records(records)
tradeoff_mc.to_csv(os.path.join(OUTDIR, "tradeoff_montecarlo_raw.csv"), index=False)

# summarize median ± IQR
tradeoff_summary = (tradeoff_mc
    .groupby(['who','frac'])
    .agg(mb_med=('mb_pg','median'),
         mb_lo =('mb_pg', lambda s: s.quantile(0.25)),
         mb_hi =('mb_pg', lambda s: s.quantile(0.75)),
         tb_med=('tb_pg','median'),
         tb_lo =('tb_pg', lambda s: s.quantile(0.25)),
         tb_hi =('tb_pg', lambda s: s.quantile(0.75)))
    .reset_index())
tradeoff_summary.to_csv(os.path.join(OUTDIR, "tradeoff_montecarlo_summary.csv"), index=False)

# quick plot
fig, ax = plt.subplots(1,2, figsize=(10,4), dpi=200)
for who, sub in tradeoff_summary.groupby('who'):
    ax[0].plot(sub['frac'], sub['mb_med'], label=who)
    ax[0].fill_between(sub['frac'], sub['mb_lo'], sub['mb_hi'], alpha=0.2)
    ax[1].plot(sub['frac'], sub['tb_med'], label=who)
    ax[1].fill_between(sub['frac'], sub['tb_lo'], sub['tb_hi'], alpha=0.2)
ax[0].axhline(0, color='k', lw=0.8); ax[1].axhline(0, color='k', lw=0.8)
ax[0].set_xlabel('Fraction retained'); ax[1].set_xlabel('Fraction retained')
ax[0].set_ylabel('Mean flux bias (Pg C yr$^{-1}$)')
ax[1].set_ylabel('Trend bias (Pg C yr$^{-2}$)')
ax[0].legend(frameon=False)
fig.suptitle('Global trade-offs (median ± IQR; Monte-Carlo)')
fig.tight_layout()
plt.savefig(os.path.join(OUTDIR, "tradeoffs_montecarlo.png"), dpi=200, bbox_inches='tight')
plt.close(fig)

# =========================================================
# B) STRATIFIED THINNING by LAT, SEASON, SECTOR (examples)
# =========================================================
# Example policy: protect winter & high-lat floats; thin ships north in summer
LAT_BANDS_EX = [(-90,-55,1.0), (-55,-35,0.4)]  # keep all south of -55; thin to 40% north of -55
SEASON_FRACS_EX = {'summer':0.4,'autumn':0.7,'winter':1.0,'spring':0.7}
SECTOR_FRACS_EX = {'Atlantic':0.8,'Indian':1.0,'Pacific':0.6}  # e.g., emphasize Indian

# Apply to SOCAT and Floats separately (you can mix & match)
soc_strat = thin_stratified(socat_fp,  LAT_BANDS_EX, SEASON_FRACS_EX, SECTOR_FRACS_EX, rng=11)
flt_strat = thin_stratified(floats_fp, LAT_BANDS_EX, SEASON_FRACS_EX, SECTOR_FRACS_EX, rng=22)

scen_strat = masked_basin_annual(truth_grid, [soc_strat, flt_strat])
mb_strat, tb_strat, _ = eval_scenario(truth_annual, scen_strat)
pd.DataFrame([{
    'mean_bias_PgCyr': to_pg(mb_strat),
    'trend_bias_PgCyr2': to_pg(tb_strat)
}]).to_csv(os.path.join(OUTDIR, "stratified_example_result.csv"), index=False)

# =================================================================
# C) TIME-BLOCK CROSS-VALIDATION (full footprints but limited years)
# =================================================================
def mask_years(fp, years_to_keep):
    return fp[fp.year.isin(years_to_keep)]

folds = [(2003,2010), (2011,2017), (2018,2024)]
cv_rows = []
for (y0,y1) in folds:
    yrs = np.arange(y0, y1+1)
    soc_fold = mask_years(socat_fp, yrs)
    flt_fold = mask_years(floats_fp, yrs)

    tf = truth_grid[truth_grid['year'].between(y0,y1)]
    ta = basin_mean_annual_from_grid_month(tf).rename(columns={'aw_flux_gC_m2_yr':'Truth'})
    scen_fold = masked_basin_annual(tf, [soc_fold, flt_fold])

    mb, tb, _ = eval_scenario(ta, scen_fold, yr0=y0, yr1=y1)
    cv_rows.append({'block':f'{y0}-{y1}', 'mean_bias_PgCyr':to_pg(mb), 'trend_bias_PgCyr2':to_pg(tb)})

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(os.path.join(OUTDIR, "timeblock_crossval.csv"), index=False)

# ==============================================================
# D) SECTOR-WISE TRADE-OFFS (global impact + sector-local view)
# ==============================================================

def in_sector(df, sector_name):
    lon_min, lon_max = SECTORS[sector_name]
    if lon_min <= lon_max:
        return df[(df['lon_bin']>=lon_min)&(df['lon_bin']<=lon_max)]
    else:
        return df[(df['lon_bin']>=lon_min)|(df['lon_bin']<=lon_max)]

# 1) Global impact: thin floats IN a sector only; all others full
fractions_sector = np.linspace(0.2, 1.0, 9)
rows_global = []
for sector in SECTORS.keys():
    # split floats fp into in-sector vs out-of-sector
    fl_in  = in_sector(floats_fp, sector)
    fl_out = floats_fp.loc[~floats_fp.index.isin(fl_in.index)]
    for f in fractions_sector:
        fl_thin_in = thin_footprint(fl_in, frac=f, by_season=True, rng=303+int(f*100))
        scen = masked_basin_annual(truth_grid, [socat_fp, fl_thin_in, fl_out])
        mb, tb, _ = eval_scenario(truth_annual, scen)
        rows_global.append({'sector':sector, 'frac':f, 'who':'Floats_in_sector',
                            'mean_bias_PgCyr':to_pg(mb), 'trend_bias_PgCyr2':to_pg(tb)})

sector_tradeoffs_global = pd.DataFrame(rows_global)
sector_tradeoffs_global.to_csv(os.path.join(OUTDIR, "sector_tradeoffs_global.csv"), index=False)

# 2) Sector-local: restrict truth & masks to that sector; vary floats there
rows_local = []
for sector in SECTORS.keys():
    tf = in_sector(truth_grid, sector)
    ta = basin_mean_annual_from_grid_month(tf).rename(columns={'aw_flux_gC_m2_yr':'Truth'})
    soc_s = in_sector(socat_fp, sector)
    flt_s = in_sector(floats_fp, sector)
    for f in fractions_sector:
        flt_thin = thin_footprint(flt_s, frac=f, by_season=True, rng=707+int(f*100))
        scen = masked_basin_annual(tf, [soc_s, flt_thin])
        mb, tb, _ = eval_scenario(ta, scen)
        rows_local.append({'sector':sector, 'frac':f, 'who':'Floats_sector_local',
                           'mean_bias_PgCyr':to_pg(mb), 'trend_bias_PgCyr2':to_pg(tb)})
sector_tradeoffs_local = pd.DataFrame(rows_local)
sector_tradeoffs_local.to_csv(os.path.join(OUTDIR, "sector_tradeoffs_local.csv"), index=False)

# quick plots for sector-local (mean bias vs fraction)
fig, ax = plt.subplots(1,2, figsize=(11,4), dpi=200)
for sector, sub in sector_tradeoffs_local.groupby('sector'):
    ax[0].plot(sub['frac'], sub['mean_bias_PgCyr'], marker='o', label=sector)
    ax[1].plot(sub['frac'], sub['trend_bias_PgCyr2'], marker='o', label=sector)
ax[0].axhline(0, color='k', lw=0.8); ax[1].axhline(0, color='k', lw=0.8)
ax[0].set_xlabel('Fraction retained (floats in sector)')
ax[1].set_xlabel('Fraction retained (floats in sector)')
ax[0].set_ylabel('Mean flux bias (Pg C yr$^{-1}$)')
ax[1].set_ylabel('Trend bias (Pg C yr$^{-2}$)')
ax[0].legend(frameon=False, ncol=3)
fig.suptitle('Sector-wise trade-offs (sector-local)')
fig.tight_layout()
plt.savefig(os.path.join(OUTDIR, "sector_tradeoffs_local.png"), dpi=200, bbox_inches='tight')
plt.close(fig)

# --------------------
# Console summary dump
# --------------------
print("\nSaved outputs in:", OUTDIR)
print(" - tradeoff_montecarlo_raw.csv")
print(" - tradeoff_montecarlo_summary.csv")
print(" - tradeoffs_montecarlo.png")
print(" - stratified_example_result.csv")
print(" - timeblock_crossval.csv")
print(" - sector_tradeoffs_global.csv")
print(" - sector_tradeoffs_local.csv")
print(" - sector_tradeoffs_local.png")



Saved outputs in: /home/ldelaigue/Documents/Python/SOCAT-OSSE-SO/SAMPLING_SUFFICIENCY
 - tradeoff_montecarlo_raw.csv
 - tradeoff_montecarlo_summary.csv
 - tradeoffs_montecarlo.png
 - stratified_example_result.csv
 - timeblock_crossval.csv
 - sector_tradeoffs_global.csv
 - sector_tradeoffs_local.csv
 - sector_tradeoffs_local.png


In [12]:
# Footprint-only, no-retrain apples-to-apples test
soc_only = masked_basin_annual(truth_grid, [socat_fp])
flt_only = masked_basin_annual(truth_grid, [floats_fp])
mb_soc, tb_soc, _ = eval_scenario(truth_annual, soc_only)
mb_flt, tb_flt, _ = eval_scenario(truth_annual, flt_only)
print("SOCAT-only bias (Pg C/yr, Pg C/yr^2):", mb_soc*7e13/1e15, tb_soc*7e13/1e15)
print("Floats-only bias (Pg C/yr, Pg C/yr^2):", mb_flt*7e13/1e15, tb_flt*7e13/1e15)

# Redistribute floats → example: protect winter + south of PF; thin over-represented corridors
LAT = [(-90,-55,1.0), (-55,-35,0.5)]
SEAS = {'summer':0.6,'autumn':0.8,'winter':1.0,'spring':0.8}
SEC  = {'Atlantic':1.0,'Indian':0.8,'Pacific':0.8}
flt_redis = thin_stratified(floats_fp, LAT, SEAS, SEC, rng=123)
scen_redis = masked_basin_annual(truth_grid, [socat_fp, flt_redis])
mb_red, tb_red, _ = eval_scenario(truth_annual, scen_redis)
print("Redistributed floats bias (Pg C/yr, Pg C/yr^2):", mb_red*7e13/1e15, tb_red*7e13/1e15)


SOCAT-only bias (Pg C/yr, Pg C/yr^2): -0.16133342445630117 -0.018633808476090297
Floats-only bias (Pg C/yr, Pg C/yr^2): -0.06844618107543875 -0.004995762845571119
Redistributed floats bias (Pg C/yr, Pg C/yr^2): -0.1188218418759466 -0.013571417224424783


In [13]:
# ============================
# Smallest fractions to meet |bias| thresholds (global)
# ============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

OUTDIR_OPT = os.path.join(OUTDIR, "optimal_fractions")
os.makedirs(OUTDIR_OPT, exist_ok=True)

# ----- User thresholds & settings -----
MEAN_THR_PG  = 0.10   # Pg C / yr
TREND_THR_PG = 0.01   # Pg C / yr^2
PROB_OK      = 0.80   # must satisfy thresholds in >= this fraction of repeats
N_REPS       = 20     # Monte-Carlo repeats per (fsoc, fflt)
F_STEPS      = np.linspace(0.10, 1.00, 19)  # search grid 0.10..1.00

# Optional cost weights (ships are $$; tweak as you like)
W_SOCAT  = 1.0
W_FLOATS = 1.0

# Optional: use expected "kept grid-months" as cost (instead of weighted fractions)
USE_EXPECTED_SAMPLES_AS_COST = False

N_SOCAT = len(socat_fp)   # grid-month rows in mask
N_FLOAT = len(floats_fp)

def eval_pair(fsoc, fflt, n_reps=N_REPS):
    """Monte-Carlo evaluation of one (fsoc, fflt) pair."""
    rows = []
    for rep in range(n_reps):
        s_mask = thin_footprint(socat_fp,  frac=fsoc, by_season=True, rng=10000 + rep)
        f_mask = thin_footprint(floats_fp, frac=fflt, by_season=True, rng=20000 + rep)
        scen   = masked_basin_annual(truth_grid, [s_mask, f_mask])
        mb, tb, _ = eval_scenario(truth_annual, scen)
        rows.append((mb, tb))
    df = pd.DataFrame(rows, columns=['mb_gCm2yr', 'tb_gCm2yr2'])
    # convert to Pg units and take absolute values
    df['mb_pg'] = df['mb_gCm2yr'].apply(to_pg).abs()
    df['tb_pg'] = df['tb_gCm2yr2'].apply(to_pg).abs()
    # quantiles for PROB_OK
    q = df.quantile(PROB_OK)
    # proportion of reps within thresholds (nice to report)
    frac_within = ((df['mb_pg'] <= MEAN_THR_PG) & (df['tb_pg'] <= TREND_THR_PG)).mean()
    return {
        'fsoc': fsoc,
        'fflt': fflt,
        'q_mb_pg': q['mb_pg'],
        'q_tb_pg': q['tb_pg'],
        'frac_within': frac_within,
        'mb_med_pg': df['mb_pg'].median(),
        'tb_med_pg': df['tb_pg'].median()
    }

# Grid search
records = []
for fs in F_STEPS:
    for ff in F_STEPS:
        rec = eval_pair(fs, ff, n_reps=N_REPS)
        if USE_EXPECTED_SAMPLES_AS_COST:
            rec['cost'] = fs*N_SOCAT + ff*N_FLOAT
        else:
            rec['cost'] = W_SOCAT*fs + W_FLOATS*ff
        records.append(rec)

opt_df = pd.DataFrame.from_records(records)
opt_df['feasible'] = (opt_df['q_mb_pg'] <= MEAN_THR_PG) & (opt_df['q_tb_pg'] <= TREND_THR_PG) & (opt_df['frac_within'] >= PROB_OK)

# Save all results
opt_df.to_csv(os.path.join(OUTDIR_OPT, "global_fraction_gridsearch.csv"), index=False)

# Pick the minimum-cost feasible solutions
feas = opt_df[opt_df['feasible']].copy().sort_values('cost')
topK = feas.head(10)
topK.to_csv(os.path.join(OUTDIR_OPT, "global_fraction_best10.csv"), index=False)

print("\nSmallest-cost feasible plans (top 10):")
print(topK[['fsoc','fflt','q_mb_pg','q_tb_pg','frac_within','cost']])

# Quick visualization of feasibility in fraction space
plt.figure(figsize=(6,5), dpi=160)
sc = plt.scatter(opt_df['fsoc'], opt_df['fflt'],
                 c=np.where(opt_df['feasible'], -opt_df['cost'], np.nan),
                 cmap='viridis', s=60, edgecolor='k')
plt.xlabel('SOCAT fraction retained')
plt.ylabel('Floats fraction retained')
plt.title('Feasible region (color = lower cost is better)')
cbar = plt.colorbar(sc)
cbar.set_label('−cost (darker = lower cost)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR_OPT, "feasible_region.png"), dpi=160)
plt.close()



Smallest-cost feasible plans (top 10):
     fsoc  fflt   q_mb_pg   q_tb_pg  frac_within  cost
285  0.85  0.10  0.074734  0.009819         0.90  0.95
286  0.85  0.15  0.075781  0.009777         0.85  1.00
304  0.90  0.10  0.069137  0.009377         0.90  1.00
287  0.85  0.20  0.077719  0.009949         0.85  1.05
305  0.90  0.15  0.071116  0.009407         0.95  1.05
323  0.95  0.10  0.071282  0.008375         0.95  1.05
288  0.85  0.25  0.079338  0.009973         0.85  1.10
306  0.90  0.20  0.072438  0.009521         0.85  1.10
324  0.95  0.15  0.073397  0.008450         0.95  1.10
342  1.00  0.10  0.071786  0.005817         1.00  1.10
